## 0.Importación librerias

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
import os
import mlflow
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import mlflow.sklearn

In [3]:
cwd = Path.cwd().resolve()
try:
    HERE = next(c for c in [cwd, *cwd.parents]
                if (c / ".git").exists() or (c / ".dvc").exists())
except StopIteration:
    raise FileNotFoundError(
        "No encuentro la raíz del repo (.git/.dvc). "
        "Abre el notebook dentro del proyecto agropredict-colombia."
    )

DATOS_PROCESADOS = HERE / "data" / "processed"
ARCHIVO_LIMPIO = DATOS_PROCESADOS / "maiz_preprocesado.parquet"

In [4]:
load_dotenv(HERE / "config" / ".env", override=True)
MLFLOW_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5001")
MLFLOW_EXPERIMENT = os.getenv("MLFLOW_EXPERIMENT", "agropredict-maiz-rendimiento")

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

print(f"MLflow apuntando a: {MLFLOW_URI}")
print(f"Experimento: {MLFLOW_EXPERIMENT}")

MLflow apuntando a: http://localhost:5001
Experimento: agropredict-pruebas-local


## 1.Configuración del experimento

In [5]:
# Tabla de features: rezago del semestre anterior (t-1) y del mismo semestre
# del año anterior (t-2), generada en 03_featuring
TIPO_REZAGO = "rezago_1_2"
ARCHIVO_FEATURES = DATOS_PROCESADOS / f"features_{TIPO_REZAGO}.parquet"

# Separación temporal de los datos
ANIOS_ENTRENAMIENTO = [2019, 2020, 2021, 2022, 2023]
ANIO_VALIDACION = 2024
ANIO_PRUEBA = 2025

# Semilla para reproducibilidad
SEMILLA = 42

# Variables del modelo nuevo: se agrega t-2 y se quita el año
NUMERICAS = ["semestre_b", "rendimiento_rezago_1", "rendimiento_rezago_2"]
CATEGORICAS = ["departamento", "sistema_productivo"]
OBJETIVO = "rendimiento"

# Variables de la referencia (especificación anterior: t-1 y año),
# evaluada sobre las mismas filas para comparar de forma justa
NUMERICAS_REFERENCIA = ["anio", "semestre_b", "rendimiento_rezago_1"]

# Resumen de la configuración activa
print("Configuración del experimento:")
print(f"  Tipo de rezago : {TIPO_REZAGO}")
print(f"  Entrenamiento  : {ANIOS_ENTRENAMIENTO}")
print(f"  Validación     : {ANIO_VALIDACION}")
print(f"  Prueba         : {ANIO_PRUEBA}")
print(f"  Semilla        : {SEMILLA}")
print(f"  Variables      : {NUMERICAS + CATEGORICAS}")
print(f"  Referencia     : {NUMERICAS_REFERENCIA + CATEGORICAS}")

Configuración del experimento:
  Tipo de rezago : rezago_1_2
  Entrenamiento  : [2019, 2020, 2021, 2022, 2023]
  Validación     : 2024
  Prueba         : 2025
  Semilla        : 42
  Variables      : ['semestre_b', 'rendimiento_rezago_1', 'rendimiento_rezago_2', 'departamento', 'sistema_productivo']
  Referencia     : ['anio', 'semestre_b', 'rendimiento_rezago_1', 'departamento', 'sistema_productivo']


In [6]:
# Cargamos la tabla de features generada en el notebook 03_featuring
base = pd.read_parquet(ARCHIVO_FEATURES)

print(f"Features cargadas: {base.shape[0]:,} filas ({TIPO_REZAGO})")
base.head()

Features cargadas: 14,935 filas (rezago_1_2)


,cod_departamento,departamento,cod_municipio,municipio,sistema_productivo,anio,semestre_b,area_cosechada,produccion,rendimiento,t,rendimiento_rezago_1,rendimiento_rezago_2
0,05,Antioquia,05002,Abejorral,tradicional,2019,0,100.0,200.0,2.0,4038,NaN,NaN
1,05,Antioquia,05002,Abejorral,tradicional,2019,1,75.0,150.0,2.0,4039,2.0,NaN
2,05,Antioquia,05002,Abejorral,tradicional,2020,0,100.0,200.0,2.0,4040,2.0,2.0
3,05,Antioquia,05002,Abejorral,tradicional,2020,1,75.0,150.0,2.0,4041,2.0,2.0
4,05,Antioquia,05002,Abejorral,tradicional,2021,0,98.0,196.0,2.0,4042,2.0,2.0


## 2. Separación temporal de los datos

Dividimos los datos respetando la línea de tiempo, según la configuración definida
al inicio:

- **Entrenamiento:** 2019–2023
- **Validación:** 2024
- **Prueba:** 2025

Antes de separar, quitamos las filas que no tienen ambos rezagos (t-1 y t-2): las
primeras apariciones de cada serie municipio-sistema, que no tienen suficiente
historia. Como el t-2 de 2019 requeriría información de 2018, ese año no aporta
filas y el entrenamiento efectivo es 2020–2023.

Todas las alternativas, incluidas la referencia y la persistencia, se entrenan y
evalúan sobre estas mismas filas, para que la comparación sea justa.

In [7]:
modelo = base.dropna(subset=["rendimiento_rezago_1", "rendimiento_rezago_2"]).copy()
print(f"Filas utilizables (con ambos rezagos): {len(modelo):,} de {len(base):,}")

# Separación temporal según la configuración del experimento
train = modelo[modelo["anio"].isin(ANIOS_ENTRENAMIENTO)]
val   = modelo[modelo["anio"] == ANIO_VALIDACION]
test  = modelo[modelo["anio"] == ANIO_PRUEBA]

print(f"\nEntrenamiento ({ANIOS_ENTRENAMIENTO[0]}-{ANIOS_ENTRENAMIENTO[-1]}): {len(train):,} filas")
print(f"Validación ({ANIO_VALIDACION}): {len(val):,} filas")
print(f"Prueba ({ANIO_PRUEBA}): {len(test):,} filas")

print("\nFilas de entrenamiento por año:")
print(train.groupby("anio").size())

Filas utilizables (con ambos rezagos): 10,782 de 14,935

Entrenamiento (2019-2023): 7,334 filas
Validación (2024): 1,713 filas
Prueba (2025): 1,735 filas

Filas de entrenamiento por año:
anio
2020    1839
2021    1916
2022    1810
2023    1769
dtype: int64


## 3.Preparar predictores (X) y objetivo (y)

Separamos lo que el modelo usa para predecir (X) de lo que queremos predecir (y,
el rendimiento).

- **Numéricos:** año, semestre y el rendimiento del pasado (rezago).
- **Categórico:** departamento, sistema productivo.

In [8]:
def separar_X_y(datos, numericas=NUMERICAS):
    """Separa predictores y objetivo según la lista de variables numéricas."""
    X = datos[numericas + CATEGORICAS]
    y = datos[OBJETIVO]
    return X, y

# Modelo nuevo: t-1, t-2, sin año
X_train, y_train = separar_X_y(train)
X_val,   y_val   = separar_X_y(val)
X_test,  y_test  = separar_X_y(test)

# Referencia: t-1 y año, sobre las mismas filas
X_train_ref, _ = separar_X_y(train, NUMERICAS_REFERENCIA)
X_val_ref,   _ = separar_X_y(val,   NUMERICAS_REFERENCIA)
X_test_ref,  _ = separar_X_y(test,  NUMERICAS_REFERENCIA)

print(f"Predictores modelo nuevo: {X_train.columns.tolist()}")
print(f"Predictores referencia : {X_train_ref.columns.tolist()}")

Predictores modelo nuevo: ['semestre_b', 'rendimiento_rezago_1', 'rendimiento_rezago_2', 'departamento', 'sistema_productivo']
Predictores referencia : ['anio', 'semestre_b', 'rendimiento_rezago_1', 'departamento', 'sistema_productivo']


### Verificación: categorías presentes en entrenamiento

El codificador aprende los departamentos y sistemas usando solo el entrenamiento.
Confirmamos que no haya ninguna categoría en validación o prueba que falte en
entrenamiento. Si faltara, el modelo estaría prediciendo un territorio del que no
vio ningún ejemplo, y habría que decidir cómo tratarlo.

In [9]:
# Verificación: ¿todas las categorías de val/test están en train?
# (Si alguna faltara, el codificador fallaría y habría que decidir cómo tratarla.)
for columna in CATEGORICAS:
    en_train = set(train[columna])
    faltan_val  = set(val[columna])  - en_train
    faltan_test = set(test[columna]) - en_train

    print(f"{columna} no vistos en entrenamiento:")
    print(f"  en validación: {faltan_val or 'ninguno'}")
    print(f"  en prueba    : {faltan_test or 'ninguno'}")

departamento no vistos en entrenamiento:
  en validación: ninguno
  en prueba    : ninguno
sistema_productivo no vistos en entrenamiento:
  en validación: ninguno
  en prueba    : ninguno


In [11]:
def crear_codificador(numericas=NUMERICAS):
    """Crea un codificador nuevo: estandariza las numéricas y aplica one-hot a las categóricas."""
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numericas),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS),
        ]
    )

# Vista previa: cómo quedan los predictores del modelo nuevo tras codificar.
# Se AJUSTA solo con entrenamiento y se APLICA a los tres conjuntos.
codificador = crear_codificador()
X_train_cod = codificador.fit_transform(X_train)
X_val_cod   = codificador.transform(X_val)
X_test_cod  = codificador.transform(X_test)

print(f"Predictores tras codificar: {X_train_cod.shape[1]} columnas")
print(f"  (numéricas: {len(NUMERICAS)}, + one-hot de departamento y sistema)")

Predictores tras codificar: 36 columnas
  (numéricas: 3, + one-hot de departamento y sistema)


# Modelo 1: Regresión Ridge (línea base lineal)

Empezamos con el modelo más simple e interpretable. Ridge es una regresión lineal
con una penalización que evita que los coeficientes crezcan demasiado, lo que la
hace más estable.

Lo usamos como **línea base lineal**: los modelos más complejos deberán superar su
desempeño para justificar su mayor complejidad. Si no lo logran, significa que el
problema es mayormente lineal.

In [12]:
# Pipeline: primero codifica las variables, luego aplica Ridge.
# Juntar codificación y modelo en un pipeline evita errores y asegura que
# val y test reciban exactamente el mismo tratamiento que train.
ridge = Pipeline(steps=[
    ("codificador", crear_codificador()),
    ("modelo", Ridge(alpha=1.0, random_state=SEMILLA)),
])

# Entrenamos con los datos de entrenamiento
ridge.fit(X_train, y_train)

# Predecimos sobre validación (2024)
pred_val_ridge = ridge.predict(X_val)

print("Ridge entrenado.")
print(f"Predicciones generadas para validación: {len(pred_val_ridge):,}")
print(f"\nEjemplo (primeras 5):")
for real, pred in zip(y_val.iloc[:5], pred_val_ridge[:5]):
    print(f"  real: {real:.2f} t/ha  |  predicho: {pred:.2f} t/ha")

Ridge entrenado.
Predicciones generadas para validación: 1,713

Ejemplo (primeras 5):
  real: 2.00 t/ha  |  predicho: 1.75 t/ha
  real: 2.00 t/ha  |  predicho: 1.87 t/ha
  real: 1.00 t/ha  |  predicho: 1.11 t/ha
  real: 1.00 t/ha  |  predicho: 1.23 t/ha
  real: 1.20 t/ha  |  predicho: 1.24 t/ha


## Evaluación: métricas de error

Medimos qué tan buenas son las predicciones con tres métricas estándar de regresión:

- **RMSE** (raíz del error cuadrático medio): error típico en t/ha. Al elevar los
  errores al cuadrado, penaliza más los errores grandes.
- **MAE** (error absoluto medio): diferencia promedio entre lo real y lo predicho,
  en t/ha. Todos los errores pesan igual, por lo que es menos sensible a valores
  extremos.
- **R²**: proporción de la variación del rendimiento que explica el modelo. Un valor
  de 1 indica predicción perfecta, y un valor negativo indica que el modelo lo hace
  peor que predecir siempre el promedio.

Las métricas se comparan además con la **persistencia** (predecir el rendimiento
del semestre anterior): un modelo solo aporta valor si mejora frente a ella.

In [21]:
def evaluar(y_real, y_pred, nombre=""):
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    mae  = mean_absolute_error(y_real, y_pred)
    r2   = r2_score(y_real, y_pred)
    print(f"{nombre}")
    print(f"  RMSE: {rmse:.3f} t/ha")
    print(f"  MAE : {mae:.3f} t/ha")
    print(f"  R²  : {r2:.3f}")
    return {"rmse": rmse, "mae": mae, "r2": r2}

# Línea base: persistencia (el rendimiento del semestre anterior)
pred_val_persistencia = val["rendimiento_rezago_1"]
metricas_persistencia = evaluar(y_val, pred_val_persistencia, "Persistencia — validación 2024")

# Evaluamos Ridge sobre validación
metricas_ridge = evaluar(y_val, pred_val_ridge, "Ridge — validación 2024")

Persistencia — validación 2024
  RMSE: 0.567 t/ha
  MAE : 0.221 t/ha
  R²  : 0.835
Ridge — validación 2024
  RMSE: 0.543 t/ha
  MAE : 0.348 t/ha
  R²  : 0.849


## ¿Qué variables pesan más? (coeficientes del Ridge)

Ridge asigna un coeficiente a cada variable: cuánto y en qué dirección influye en el
rendimiento. Como el codificador estandariza las variables numéricas (las deja en la
misma escala), sus coeficientes se pueden comparar directamente: por ejemplo, el peso
del rendimiento del semestre anterior (t-1) frente al del mismo semestre del año
anterior (t-2).

Un coeficiente positivo empuja el rendimiento hacia arriba; uno negativo, hacia abajo.
El tamaño (sin signo) indica la fuerza.

En las variables categóricas (departamento y sistema productivo), cada coeficiente
indica cuánto se desplaza el rendimiento al pertenecer a esa categoría.

## Desempeño de Ridge

Resultados en validación (2024), sobre las filas con ambos rezagos:

- **RMSE = 0.543 t/ha:** el error típico ronda 0.54 toneladas por hectárea, un 4 %
  menor que el de la persistencia.
- **MAE = 0.348 t/ha:** en promedio, la predicción se desvía 0.35 toneladas del valor
  real, bastante más que la persistencia (0.22).
- **R² = 0.849:** el modelo explica el 85 % de la variación del rendimiento.

Ridge mejora frente a la persistencia en RMSE y R², pero empeora en MAE. La razón es
una característica de los datos: en el 42.3 % de las filas de validación el
rendimiento es exactamente igual al del semestre anterior. En esas filas la
persistencia acierta con error cero, mientras que Ridge, al combinar t-1 y t-2 con
coeficientes, nunca reproduce exactamente el valor anterior. Así comete pequeños
errores en casi todas las filas (sube el MAE), pero suaviza los cambios grandes
(baja el RMSE).

Por esto se usa el **RMSE como métrica principal** de comparación: refleja mejor el
desempeño en los periodos donde el rendimiento realmente cambia, que son los
relevantes para la planeación.

In [16]:
# El codificador de `ridge` ya estandariza las numéricas, así que sus
# coeficientes son comparables entre sí sin entrenar otro modelo
nombres = ridge.named_steps["codificador"].get_feature_names_out()
coeficientes = ridge.named_steps["modelo"].coef_

importancia = (pd.DataFrame({"variable": nombres, "coeficiente": coeficientes})
                 .assign(peso=lambda d: d["coeficiente"].abs())
                 .sort_values("peso", ascending=False))

print("Variables con mayor peso:")
print(importancia[["variable", "coeficiente"]].head(12).to_string(index=False))

Variables con mayor peso:
                           variable  coeficiente
          num__rendimiento_rezago_1     0.998736
             cat__departamento_Meta     0.479037
cat__sistema_productivo_tecnificado     0.390995
cat__sistema_productivo_tradicional    -0.390995
  cat__departamento_Valle del Cauca     0.321388
            cat__departamento_Chocó    -0.278301
           cat__departamento_Nariño    -0.256132
        cat__departamento_Atlántico    -0.252640
         cat__departamento_Casanare     0.247980
          cat__departamento_Vichada     0.234446
          cat__departamento_Guainía    -0.210513
          num__rendimiento_rezago_2     0.208358


- **Los rezagos**: el rendimiento del semestre anterior (t-1) es la variable más fuerte
  (coeficiente ~1.00), cerca de cinco veces el peso del mismo semestre del año anterior
  (t-2, ~0.21). El rendimiento reciente es el mejor predictor; la información
  estacional aporta, pero de forma secundaria. Como ambos rezagos están muy
  correlacionados, el reparto exacto de peso entre ellos debe interpretarse con cautela.

- **El sistema productivo**: tecnificado empuja hacia arriba (+0.39) y tradicional hacia
  abajo (−0.39). Coincide con el EDA, donde el tecnificado rendía más del doble.

- **Departamentos**: algunos departamentos tienen peso propio: Meta (+0.48), Valle del
  Cauca (+0.32), Casanare (+0.25) y Vichada (+0.23) hacia arriba; Chocó (−0.28),
  Nariño (−0.26), Atlántico (−0.25) y Guainía (−0.21) hacia abajo.

Los coeficientes de las variables numéricas (por desviación estándar) y los de las
categóricas (por pertenencia a la categoría) no están en la misma escala, por lo que su
orden es una guía y no una comparación estricta.

## Registro en MLflow: Ridge

Envolvemos el entrenamiento en un *run* de MLflow. Todo lo que ocurra dentro del
bloque `with mlflow.start_run(...)` queda asociado a ese run: los parámetros que
usamos, las métricas que obtuvimos y el modelo entrenado. Esto nos permite comparar
este modelo con los siguientes en la interfaz de MLflow.

In [22]:
def registrar_run(nombre_modelo, metricas, numericas=NUMERICAS, parametros=None, modelo=None):
    """Registra en MLflow la configuración, métricas y (opcionalmente) el modelo entrenado."""
    with mlflow.start_run(run_name=f"{nombre_modelo.lower()}_{TIPO_REZAGO}"):

        # Parámetros: configuración del experimento
        mlflow.log_param("modelo", nombre_modelo)
        mlflow.log_param("tipo_rezago", TIPO_REZAGO)
        mlflow.log_param("variables", numericas + CATEGORICAS)
        mlflow.log_param("anios_entrenamiento", f"{ANIOS_ENTRENAMIENTO[0]}-{ANIOS_ENTRENAMIENTO[-1]}")
        mlflow.log_param("anio_validacion", ANIO_VALIDACION)
        mlflow.log_param("filas_entrenamiento", len(train))
        mlflow.log_param("filas_validacion", len(val))
        if parametros:
            mlflow.log_params(parametros)

        # Métricas de validación
        mlflow.log_metric("rmse_val", metricas["rmse"])
        mlflow.log_metric("mae_val", metricas["mae"])
        mlflow.log_metric("r2_val", metricas["r2"])

        # Pipeline completo (codificador + modelo), si aplica.
        # Se usa pickle porque el formato por defecto no admite modelos de árboles.
        if modelo is not None:
            mlflow.sklearn.log_model(modelo, name="modelo", serialization_format="pickle")

    print(f"Run de {nombre_modelo} registrado en MLflow.")


# Línea base: no tiene modelo entrenado, solo métricas
registrar_run("Persistencia", metricas_persistencia, numericas=["rendimiento_rezago_1"])

# Ridge
registrar_run("Ridge", metricas_ridge, parametros={"alpha": 1.0}, modelo=ridge)

2026/11/13 20:30:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run persistencia_rezago_1_2 at: http://localhost:5001/#/experiments/2/runs/819f801d621340c4ba034f559e267ff7
🧪 View experiment at: http://localhost:5001/#/experiments/2
Run de Persistencia registrado en MLflow.
🏃 View run ridge_rezago_1_2 at: http://localhost:5001/#/experiments/2/runs/7698764987284808bc8feea5cc6aed6b
🧪 View experiment at: http://localhost:5001/#/experiments/2
Run de Ridge registrado en MLflow.


# Modelo 2: Random Forest

Random Forest combina muchos árboles de decisión entrenados sobre muestras distintas
de los datos y promedia sus resultados. Esto le permite capturar **relaciones no
lineales e interacciones** entre variables que un modelo lineal como Ridge no puede.

Si logra superarlo, significa que hay estructura no lineal que vale la pena aprovechar. Si apenas lo iguala, confirma que el problema es mayormente lineal.

Usamos el mismo codificador (one-hot para las categóricas), aunque los árboles no
necesitan estandarizar las variables numéricas.

In [23]:
rf = Pipeline(steps=[
    ("codificador", crear_codificador()),
    ("modelo", RandomForestRegressor(
        n_estimators=200,       # número de árboles
        max_depth=None,         # sin límite de profundidad (se controla con los otros parámetros)
        min_samples_leaf=5,     # mínimo de observaciones por hoja: frena el sobreajuste
        random_state=SEMILLA,
        n_jobs=-1,              # usa todos los núcleos disponibles
    )),
])

rf.fit(X_train, y_train)
pred_val_rf = rf.predict(X_val)

In [24]:
metricas_rf = evaluar(y_val, pred_val_rf, "Random Forest — validación 2024")

Random Forest — validación 2024
  RMSE: 0.530 t/ha
  MAE : 0.279 t/ha
  R²  : 0.856


## Desempeño de Random Forest

- **RMSE = 0.530 t/ha:** el error típico ronda media tonelada por hectárea. Es el más
  bajo hasta ahora, un 6.5 % por debajo de la persistencia.
- **MAE = 0.279 t/ha:** en promedio, la predicción se desvía cerca de 0.28 toneladas
  del valor real. Mejora frente a Ridge, pero sigue por encima de la persistencia.
- **R² = 0.856:** el modelo explica el 86 % de la variación del rendimiento.

Random Forest mejora frente a Ridge en las tres métricas, sobre todo en el MAE (de
0.348 a 0.279). Al ser un modelo basado en árboles, puede predecir valores muy
cercanos al rezago cuando la serie es estable, algo que Ridge no logra por ser lineal.

La brecha entre el RMSE y el MAE indica que el error se concentra en pocos casos
grandes: en la mayoría de las filas la predicción es cercana al valor real, pero
algunos cambios bruscos de rendimiento siguen costándole. La persistencia conserva el
menor MAE por las filas en las que el rendimiento se repite exactamente.

## Registro en MLflow: Random Forest

In [25]:
registrar_run(
    "RandomForest", metricas_rf,
    parametros={"n_estimators": 200, "max_depth": "None", "min_samples_leaf": 5},
    modelo=rf,
)

2026/11/13 20:30:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run randomforest_rezago_1_2 at: http://localhost:5001/#/experiments/2/runs/d764de0a078441f0b677a1ba253ff0bb
🧪 View experiment at: http://localhost:5001/#/experiments/2
Run de RandomForest registrado en MLflow.


# Modelo 3: XGBoost

XGBoost también usa árboles, pero de forma distinta a Random Forest. En vez de
construirlos en paralelo y promediarlos, los construye **en secuencia**: cada árbol
nuevo se concentra en corregir los errores que dejaron los anteriores. Así va
afinando la predicción paso a paso.

Esta técnica (llamada *boosting*) suele lograr el mejor desempeño, pero es más
sensible a su configuración y más propensa a sobreajustar si no se controla.

In [26]:
xgb = Pipeline(steps=[
    ("codificador", crear_codificador()),
    ("modelo", XGBRegressor(
        n_estimators=300,        # número de árboles (pasos de corrección)
        learning_rate=0.05,      # cuánto corrige cada árbol: bajo = más gradual y estable
        max_depth=5,             # profundidad de cada árbol: frena el sobreajuste
        subsample=0.8,           # cada árbol usa el 80% de los datos: añade robustez
        colsample_bytree=0.8,    # cada árbol usa el 80% de las variables
        random_state=SEMILLA,
        n_jobs=-1,
    )),
])

xgb.fit(X_train, y_train)
pred_val_xgb = xgb.predict(X_val)

In [27]:
metricas_xgb = evaluar(y_val, pred_val_xgb, "XGBoost — validación 2024")

XGBoost — validación 2024
  RMSE: 0.519 t/ha
  MAE : 0.278 t/ha
  R²  : 0.862


### Desempeño de XGBoost

- **RMSE = 0.519 t/ha:** el error típico ronda media tonelada por hectárea. Es el más
  bajo de los tres modelos, un 8.5 % por debajo de la persistencia.
- **MAE = 0.278 t/ha:** en promedio, la predicción se desvía cerca de 0.28 toneladas
  del valor real, prácticamente igual que Random Forest.
- **R² = 0.862:** el modelo explica el 86 % de la variación del rendimiento.

XGBoost obtiene el mejor RMSE y R², mientras que su MAE es casi idéntico al de Random
Forest: su ventaja está en reducir los errores grandes, no los pequeños. Al igual que
en los otros modelos, la brecha entre el RMSE y el MAE indica que el error se concentra
en pocos cambios bruscos de rendimiento, y la persistencia conserva el menor MAE por las
filas en las que el rendimiento se repite exactamente.

## Registro en MLflow: XGBoost

In [28]:
registrar_run(
    "XGBoost", metricas_xgb,
    parametros={
        "n_estimators": 300, "learning_rate": 0.05, "max_depth": 5,
        "subsample": 0.8, "colsample_bytree": 0.8,
    },
    modelo=xgb,
)

2026/11/13 20:33:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run xgboost_rezago_1_2 at: http://localhost:5001/#/experiments/2/runs/32a9a3246f8e47b2a54f4895bca2061b
🧪 View experiment at: http://localhost:5001/#/experiments/2
Run de XGBoost registrado en MLflow.
